In [1]:
import os
import re
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold
from sklearn.metrics import f1_score
from xgboost import XGBClassifier
import lightgbm as lgb
import warnings
warnings.filterwarnings("ignore")

INPUT_DIR = "./dataset/public"
OUTPUT_PATH = "./working/submission.csv"

CLEAN_COLS = ["transient_class", "host_environment", "spectral_regime"]
DERIVED_COLS = ["variability_pattern", "distance_bin", "energy_tier", "followup_protocol", "precursor_category"]
LABEL_COLS = CLEAN_COLS + DERIVED_COLS
N_FOLDS = 5
SVD_COMPONENTS = 25

def extract_numeric_features(df):
    df = df.copy()
    df['redshift'] = df['narrative'].str.extract(r'z\s*=\s*(\d+(?:\.\d+)?)').astype(float)
    df['luminosity'] = df['narrative'].str.extract(r'log L\s*=\s*(\d+(?:\.\d+)?)').astype(float)
    df['redshift'] = df['redshift'].fillna(-999)
    df['luminosity'] = df['luminosity'].fillna(-999)
    return df

def main():
    print("Step 1: Loading Data & Engineering Features...")
    train_df = pd.read_csv(os.path.join(INPUT_DIR, "train.csv"))
    test_df = pd.read_csv(os.path.join(INPUT_DIR, "test.csv"))
    
    train_df = extract_numeric_features(train_df)
    test_df = extract_numeric_features(test_df)
    
    # Extract SVD Embeddings from the Text Narrative
    print(f"Step 2: Compressing Text Narratives via TF-IDF + SVD ({SVD_COMPONENTS} components)...")
    tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=10000, stop_words='english')
    svd = TruncatedSVD(n_components=SVD_COMPONENTS, random_state=42)
    
    train_tfidf = tfidf.fit_transform(train_df['narrative'])
    test_tfidf = tfidf.transform(test_df['narrative'])
    
    train_svd = svd.fit_transform(train_tfidf)
    test_svd = svd.transform(test_tfidf)
    
    svd_cols = [f"svd_{i}" for i in range(SVD_COMPONENTS)]
    train_df[svd_cols] = train_svd
    test_df[svd_cols] = test_svd

    print("Step 3: Training NLP Extractors for Clean Labels...")
    for col in CLEAN_COLS:
        pipeline = make_pipeline(
            TfidfVectorizer(ngram_range=(1, 3), max_features=5000), 
            LogisticRegression(C=10, max_iter=1000, random_state=42)
        )
        pipeline.fit(train_df['narrative'], train_df[col])
        test_df[col] = pipeline.predict(test_df['narrative'])

    label_encoders = {}
    for col in LABEL_COLS:
        le = LabelEncoder()
        le.fit(train_df[col])
        train_df[col + "_enc"] = le.transform(train_df[col])
        if col in CLEAN_COLS:
            test_df[col + "_enc"] = le.transform(test_df[col])
        label_encoders[col] = le

    print("\nStep 4: Training XGBoost & LightGBM Ensembles for Derived Labels...")
    features = [col + "_enc" for col in CLEAN_COLS] + ["redshift", "luminosity"] + svd_cols
    
    test_predictions = {col: np.zeros((len(test_df), len(label_encoders[col].classes_))) for col in DERIVED_COLS}
    
    pairs = train_df[['transient_class', 'host_environment']].drop_duplicates().values.tolist()
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
    
    for fold, (train_idx_pairs, val_idx_pairs) in enumerate(kf.split(pairs)):
        print(f"\n--- FOLD {fold + 1}/{N_FOLDS} ---")
        val_pairs = [pairs[i] for i in val_idx_pairs]
        val_mask = train_df.apply(lambda row: [row['transient_class'], row['host_environment']] in val_pairs, axis=1)
        
        X_train = train_df[~val_mask][features]
        X_val = train_df[val_mask][features]
        
        for col in DERIVED_COLS:
            y_train = train_df[~val_mask][col + "_enc"]
            y_val = train_df[val_mask][col + "_enc"]
            
            # MODEL 1: XGBoost (Depth-wise, highly regularized child weights)
            xgb_model = XGBClassifier(
                n_estimators=400,
                max_depth=3,
                learning_rate=0.04,
                subsample=0.8,
                colsample_bytree=0.7,
                min_child_weight=5,  # Force algorithm to ignore noisy outliers
                eval_metric="mlogloss",
                early_stopping_rounds=40,
                random_state=42 + fold
            )
            xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
            
            # MODEL 2: LightGBM (Leaf-wise, highly regularized leaf limits)
            lgb_model = lgb.LGBMClassifier(
                n_estimators=400,
                max_depth=3,
                learning_rate=0.04,
                subsample=0.8,
                colsample_bytree=0.7,
                min_child_samples=30, # Force algorithm to ignore noisy outliers
                random_state=42 + fold,
                verbose=-1
            )
            # LightGBM requires early stopping via callbacks in recent versions
            lgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(40, verbose=False)])
            
            # Ensemble Predictions (50/50 blend)
            val_preds_proba = (xgb_model.predict_proba(X_val) + lgb_model.predict_proba(X_val)) / 2
            val_preds = np.argmax(val_preds_proba, axis=1)
            
            val_f1 = f1_score(y_val, val_preds, average="macro")
            print(f"  [{col}] Ensemble Val Macro F1: {val_f1:.4f}")
            
            # Accumulate blended test predictions
            test_pred_xgb = xgb_model.predict_proba(test_df[features])
            test_pred_lgb = lgb_model.predict_proba(test_df[features])
            test_predictions[col] += ((test_pred_xgb + test_pred_lgb) / 2) / N_FOLDS

    print("\nStep 5: Compiling Final Submission...")
    submission_df = pd.DataFrame({'id': test_df['id']})
    
    for col in CLEAN_COLS:
        submission_df[col] = test_df[col]
        
    for col in DERIVED_COLS:
        final_class_indices = np.argmax(test_predictions[col], axis=1)
        submission_df[col] = label_encoders[col].inverse_transform(final_class_indices)
        
    submission_df = submission_df[["id"] + LABEL_COLS]
    os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
    submission_df.to_csv(OUTPUT_PATH, index=False)
    print(f"Done! Predictions saved to {OUTPUT_PATH}")

if __name__ == "__main__":
    main()

Step 1: Loading Data & Engineering Features...
Step 2: Compressing Text Narratives via TF-IDF + SVD (25 components)...
Step 3: Training NLP Extractors for Clean Labels...

Step 4: Training XGBoost & LightGBM Ensembles for Derived Labels...

--- FOLD 1/5 ---
  [variability_pattern] Ensemble Val Macro F1: 0.7179
  [distance_bin] Ensemble Val Macro F1: 0.6395
  [energy_tier] Ensemble Val Macro F1: 0.7353
  [followup_protocol] Ensemble Val Macro F1: 0.7123
  [precursor_category] Ensemble Val Macro F1: 0.5255

--- FOLD 2/5 ---
  [variability_pattern] Ensemble Val Macro F1: 0.6703
  [distance_bin] Ensemble Val Macro F1: 0.6539
  [energy_tier] Ensemble Val Macro F1: 0.6913
  [followup_protocol] Ensemble Val Macro F1: 0.6940
  [precursor_category] Ensemble Val Macro F1: 0.5366

--- FOLD 3/5 ---
  [variability_pattern] Ensemble Val Macro F1: 0.7008
  [distance_bin] Ensemble Val Macro F1: 0.6454
  [energy_tier] Ensemble Val Macro F1: 0.6632
  [followup_protocol] Ensemble Val Macro F1: 0.7023
  [